# Unified EDA, Data Preprocessing, and Feature Engineering Notebook

**Project:** Diabetes Prediction and Explainable AI  
**Authors:** Mohammed Sherbini, Mohamed Hany, Hany Mohammed, Ibrahim Atef

This notebook consolidates the **EDA**, **data preprocessing**, **class-balancing**, **scaling**, **feature ranking**, **feature selection**, and **feature-engineering** work that appeared across the uploaded project notebooks:

- `Diabetes_XAI_hany_FINAL`
- `Final_XAI`
- `Ibrahim Atef_DECISION TREE`
- `Ibrahim Atef_LOGISTIC Regression`
- `Ibrahim Atef_RANDOM_FORESt`
- `Mohammed_Sherbini_ExtraTrees`
- `Mohammed_Sherbini_LightGBM`
- `Mohammed_Sherbini_XGBoost`

The goal is to keep all dataset preparation work in one clean, reproducible place before model training and XAI.

## 1. Setup

Run this cell first. It imports the libraries used for EDA, preprocessing, feature engineering, feature selection, and SMOTE balancing.

In [ ]:
# If needed, uncomment these installs in Colab/Jupyter:
# !pip install -q imbalanced-learn seaborn scikit-learn

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import f_classif, mutual_info_classif, SelectKBest
from sklearn.ensemble import RandomForestClassifier

try:
    from imblearn.over_sampling import SMOTE, BorderlineSMOTE
    IMBLEARN_AVAILABLE = True
except Exception as e:
    IMBLEARN_AVAILABLE = False
    print("imbalanced-learn is not available. Install it with: pip install imbalanced-learn")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)


## 2. Load the BRFSS Diabetes Dataset

All uploaded notebooks used the **BRFSS 2015 diabetes health indicators** dataset. Put the CSV in the same folder as this notebook, or update `CSV_PATH` manually.

In [ ]:
# Common paths used across the notebooks
candidate_paths = [
    "/content/diabetes_012_health_indicators_BRFSS2015.csv",
    "diabetes_012_health_indicators_BRFSS2015.csv",
    "./diabetes_012_health_indicators_BRFSS2015.csv",
    "../diabetes_012_health_indicators_BRFSS2015.csv",
]

CSV_PATH = None
for path in candidate_paths:
    if os.path.exists(path):
        CSV_PATH = path
        break

if CSV_PATH is None:
    raise FileNotFoundError(
        "Dataset CSV not found. Put diabetes_012_health_indicators_BRFSS2015.csv next to this notebook "
        "or set CSV_PATH manually."
    )

df_raw = pd.read_csv(CSV_PATH)
print("Loaded from:", CSV_PATH)
print("Raw shape:", df_raw.shape)
display(df_raw.head())


## 3. Standardize Column Names and Preserve Original Format

Some notebooks used the original column names, while the XGBoost/LightGBM/ExtraTrees notebooks converted names to lowercase. This notebook keeps both versions:

- `df_original`: original column names such as `Diabetes_012`, `HighBP`, `BMI`
- `df_lower`: lowercase names such as `diabetes_012`, `highbp`, `bmi`

In [ ]:
df_original = df_raw.copy()
df_original.columns = df_original.columns.str.strip()

df_lower = df_original.copy()
df_lower.columns = df_lower.columns.str.strip().str.lower()

print("Original columns:")
print(df_original.columns.tolist())

print("\nLowercase columns:")
print(df_lower.columns.tolist())


## 4. Dataset Overview EDA

This combines the overview checks used in the notebooks: shape, column list, data types, descriptive statistics, missing values, duplicate rows, and preview rows.

In [ ]:
print("Shape:", df_original.shape)
print("\nColumns:")
print(df_original.columns.tolist())

print("\nInfo:")
display(df_original.info())

print("\nFirst rows:")
display(df_original.head())

print("\nDescriptive statistics:")
display(df_original.describe().T)

missing_table = (
    pd.DataFrame({
        "missing_count": df_original.isnull().sum(),
        "missing_percent": (df_original.isnull().mean() * 100).round(4)
    })
    .sort_values("missing_count", ascending=False)
)
print("\nMissing values:")
display(missing_table)

print("\nDuplicate rows before preprocessing:", df_original.duplicated().sum())


## 5. Target Exploration: Multiclass and Binary Diabetes Targets

The notebooks used two target formulations:

1. **Multiclass target:** `Diabetes_012` / `diabetes_012`  
   - 0 = no diabetes
   - 1 = prediabetes
   - 2 = diabetes

2. **Binary target:** `Diabetes_binary`  
   - 0 = no diabetes
   - 1 = prediabetes or diabetes

In [ ]:
# Multiclass target table
if "Diabetes_012" in df_original.columns:
    target_col_original = "Diabetes_012"
elif "diabetes_012" in df_original.columns:
    target_col_original = "diabetes_012"
else:
    raise KeyError("Could not find Diabetes_012 target column.")

target_counts = df_original[target_col_original].value_counts().sort_index()
target_percent = (target_counts / target_counts.sum() * 100).round(2)
target_table = pd.DataFrame({"count": target_counts, "percent": target_percent})
print("Multiclass target distribution:")
display(target_table)

# Binary target creation used in Diabetes_XAI_hany_FINAL and Random Forest notebook
df_binary = df_original.copy()
df_binary["Diabetes_binary"] = (df_binary[target_col_original] > 0).astype(int)

binary_counts = df_binary["Diabetes_binary"].value_counts().sort_index()
binary_percent = (binary_counts / binary_counts.sum() * 100).round(2)
binary_table = pd.DataFrame({"count": binary_counts, "percent": binary_percent})
print("\nBinary target distribution:")
display(binary_table)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(x=target_table.index.astype(str), y=target_table["count"], ax=axes[0])
axes[0].set_title("Multiclass Diabetes_012 Distribution")
axes[0].set_xlabel("Class")
axes[0].set_ylabel("Count")

sns.barplot(x=binary_table.index.astype(str), y=binary_table["count"], ax=axes[1])
axes[1].set_title("Binary Diabetes Distribution")
axes[1].set_xlabel("Class")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()


## 6. Outlier, Skewness, and Summary Checks

The Final_XAI notebook specifically checked BMI IQR outliers and skewness for BMI, Age, physical health, and mental health.

In [ ]:
continuous_check_cols = [col for col in ["BMI", "Age", "PhysHlth", "MentHlth"] if col in df_original.columns]

print("Skewness for selected continuous/ordinal features:")
display(df_original[continuous_check_cols].skew().rename("skewness").to_frame())

if "BMI" in df_original.columns:
    Q1 = df_original["BMI"].quantile(0.25)
    Q3 = df_original["BMI"].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    bmi_outliers = df_original[(df_original["BMI"] < lower) | (df_original["BMI"] > upper)]

    print(f"BMI Q1={Q1:.3f}, Q3={Q3:.3f}, IQR={IQR:.3f}")
    print(f"BMI lower bound={lower:.3f}, upper bound={upper:.3f}")
    print("BMI outlier rows:", len(bmi_outliers))

    plt.figure(figsize=(8, 4))
    sns.boxplot(x=df_original["BMI"])
    plt.title("BMI Boxplot and IQR Outliers")
    plt.tight_layout()
    plt.show()


## 7. Group Statistics by Diabetes Class

Several notebooks compared feature averages by diabetes class. This is useful for understanding how health indicators differ across no-diabetes, prediabetes, and diabetes groups.

In [ ]:
group_cols = [col for col in ["BMI", "Age", "PhysHlth", "MentHlth", "GenHlth"] if col in df_original.columns]

if group_cols:
    print("Mean values by multiclass diabetes target:")
    display(df_original.groupby(target_col_original)[group_cols].mean().round(3))

    print("\nMedian values by multiclass diabetes target:")
    display(df_original.groupby(target_col_original)[group_cols].median().round(3))


## 8. Correlation Analysis

This consolidates correlation work from the notebooks:

- full correlation matrix heatmap
- correlation with the multiclass target
- correlation with the binary target
- top absolute target correlations

In [ ]:
# Multiclass correlation matrix
plt.figure(figsize=(13, 10))
corr_multi = df_original.corr(numeric_only=True)
sns.heatmap(corr_multi, cmap="coolwarm", center=0, linewidths=0.2, cbar_kws={"shrink": 0.75})
plt.title("Correlation Matrix - Original Features")
plt.tight_layout()
plt.show()

print("Correlation with multiclass target:")
target_corr_multi = corr_multi[target_col_original].drop(target_col_original).sort_values(ascending=False)
display(target_corr_multi.to_frame("corr_with_Diabetes_012"))

# Binary correlation heatmap and top features
corr_binary = df_binary.drop(columns=[target_col_original], errors="ignore").corr(numeric_only=True)
if "Diabetes_binary" in corr_binary.columns:
    plt.figure(figsize=(13, 10))
    sns.heatmap(corr_binary, cmap="coolwarm", center=0, linewidths=0.2, cbar_kws={"shrink": 0.75})
    plt.title("Correlation Matrix - Binary Target Version")
    plt.tight_layout()
    plt.show()

    target_corr_binary = corr_binary["Diabetes_binary"].drop("Diabetes_binary").abs().sort_values(ascending=False)
    print("Top 15 features by absolute correlation with binary diabetes target:")
    display(target_corr_binary.head(15).to_frame("abs_corr_with_Diabetes_binary"))


## 9. Feature Distributions and Class Relationships

This section combines the KDE, boxplot, and categorical countplot EDA used in the notebooks.

In [ ]:
# KDE distributions by multiclass target
for col in [c for c in ["BMI", "Age", "PhysHlth", "MentHlth"] if c in df_original.columns]:
    plt.figure(figsize=(8, 4))
    sns.kdeplot(data=df_original, x=col, hue=target_col_original, fill=True, common_norm=False, alpha=0.35)
    plt.title(f"{col} Distribution by Diabetes_012 Class")
    plt.tight_layout()
    plt.show()

# Boxplots for continuous columns
for col in [c for c in ["BMI", "MentHlth", "PhysHlth"] if c in df_original.columns]:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=target_col_original, y=col, data=df_original)
    plt.title(f"{col} by Diabetes_012 Class")
    plt.tight_layout()
    plt.show()

# Categorical/binary countplots used in Final_XAI
cat_cols = [c for c in ["HighBP", "HighChol", "PhysActivity", "DiffWalk", "HeartDiseaseorAttack"] if c in df_original.columns]
for col in cat_cols:
    plt.figure(figsize=(7, 4))
    sns.countplot(x=col, hue=target_col_original, data=df_original)
    plt.title(f"{col} vs Diabetes_012")
    plt.tight_layout()
    plt.show()


## 10. Cleaning Variants Used Across Notebooks

The notebooks used these cleaning decisions:

- strip whitespace from column names
- drop missing rows in some notebooks
- remove duplicate rows in Final_XAI
- optionally remove BMI IQR outliers in Final_XAI
- convert features to numeric and median-impute in XGBoost/LightGBM/ExtraTrees notebooks

The function below makes those steps explicit and configurable.

In [ ]:
def clean_diabetes_data(
    data,
    drop_missing=False,
    drop_duplicates=True,
    remove_bmi_outliers=False,
    target_col="Diabetes_012"
):
    # Apply cleaning choices used across the uploaded notebooks.
    df = data.copy()
    df.columns = df.columns.str.strip()

    print("Initial shape:", df.shape)

    if drop_missing:
        df = df.dropna()
        print("After dropping missing rows:", df.shape)

    if drop_duplicates:
        before = df.duplicated().sum()
        df = df.drop_duplicates().reset_index(drop=True)
        print(f"Removed duplicate rows: {before}")
        print("After duplicate removal:", df.shape)

    if remove_bmi_outliers and "BMI" in df.columns:
        Q1 = df["BMI"].quantile(0.25)
        Q3 = df["BMI"].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        before = len(df)
        df = df[(df["BMI"] >= lower) & (df["BMI"] <= upper)].reset_index(drop=True)
        print(f"Removed BMI outlier rows: {before - len(df)}")
        print("After BMI outlier removal:", df.shape)

    return df

# Standard clean version: duplicate removal only
# Final_XAI-like stricter version: duplicate removal + BMI IQR outlier removal

df_clean = clean_diabetes_data(df_original, drop_missing=False, drop_duplicates=True, remove_bmi_outliers=False)
df_clean_no_bmi_outliers = clean_diabetes_data(df_original, drop_missing=False, drop_duplicates=True, remove_bmi_outliers=True)


## 11. Unified Feature Engineering

This section includes the feature-engineering formulas used across the notebooks:

- `BMI_Age`, `BMI_x_Age`, `BMI_Age_Interaction`
- `Risk_score`, `Health_Risk_Score`, `Cardiometabolic_Risk_Score`
- `Lifestyle_score`, `Lifestyle_Score`, `Lifestyle_Risk_Score`
- `No_Physical_Activity`, `Low_Fruit_Intake`, `Low_Veggie_Intake`
- `Poor_Health_Days`
- `Hlth_burden`
- `Health_Limitation_Score`
- `SES`

Some names are aliases from different notebooks. They are preserved here so model-specific notebooks can reproduce their exact feature sets.

In [ ]:
def add_engineered_features(data):
    # Add all engineered features found across the uploaded notebooks.
    df = data.copy()
    df.columns = df.columns.str.strip()

    # 1. BMI x Age interactions / aliases
    if {"BMI", "Age"}.issubset(df.columns):
        df["BMI_Age"] = df["BMI"] * df["Age"]                      # Final_XAI / Decision Tree
        df["BMI_x_Age"] = df["BMI"] * df["Age"]                    # Diabetes_XAI_hany_FINAL
        df["BMI_Age_Interaction"] = df["BMI"] * df["Age"]          # Random Forest

    # 2. Cardiometabolic / clinical risk scores
    risk_cols_hany = [c for c in ["HighBP", "HighChol", "Stroke", "HeartDiseaseorAttack", "DiffWalk"] if c in df.columns]
    risk_cols_rf = [c for c in ["HighBP", "HighChol", "HeartDiseaseorAttack", "Stroke"] if c in df.columns]

    if risk_cols_hany:
        df["Risk_score"] = df[risk_cols_hany].fillna(0).sum(axis=1)            # Diabetes_XAI_hany_FINAL
        df["Health_Risk_Score"] = df[risk_cols_hany].fillna(0).sum(axis=1)     # Decision Tree

    if risk_cols_rf:
        df["Cardiometabolic_Risk_Score"] = df[risk_cols_rf].fillna(0).sum(axis=1)  # Random Forest

    # 3. Lifestyle scores and inverse lifestyle indicators
    positive_lifestyle = [c for c in ["PhysActivity", "Fruits", "Veggies"] if c in df.columns]
    negative_lifestyle = [c for c in ["Smoker", "HvyAlcoholConsump"] if c in df.columns]

    if "PhysActivity" in df.columns:
        df["No_Physical_Activity"] = 1 - df["PhysActivity"]
    if "Fruits" in df.columns:
        df["Low_Fruit_Intake"] = 1 - df["Fruits"]
    if "Veggies" in df.columns:
        df["Low_Veggie_Intake"] = 1 - df["Veggies"]

    if positive_lifestyle or negative_lifestyle:
        pos_sum = df[positive_lifestyle].fillna(0).sum(axis=1) if positive_lifestyle else 0
        neg_sum = df[negative_lifestyle].fillna(0).sum(axis=1) if negative_lifestyle else 0
        df["Lifestyle_score"] = pos_sum - df[[c for c in ["Smoker"] if c in df.columns]].fillna(0).sum(axis=1)  # Hany-style
        df["Lifestyle_Score"] = pos_sum - neg_sum  # Decision Tree-style

        lifestyle_risk_cols = [c for c in ["Smoker", "No_Physical_Activity", "Low_Fruit_Intake", "Low_Veggie_Intake", "HvyAlcoholConsump"] if c in df.columns]
        if lifestyle_risk_cols:
            df["Lifestyle_Risk_Score"] = df[lifestyle_risk_cols].fillna(0).sum(axis=1)  # Random Forest-style

    # 4. Health burden / limitation scores
    if {"MentHlth", "PhysHlth"}.issubset(df.columns):
        df["Poor_Health_Days"] = df["MentHlth"].fillna(0) + df["PhysHlth"].fillna(0)

    if {"GenHlth", "MentHlth", "PhysHlth"}.issubset(df.columns):
        df["Hlth_burden"] = df["GenHlth"] + df["MentHlth"] / 30 + df["PhysHlth"] / 30

    health_limitation_cols = [c for c in ["GenHlth", "DiffWalk", "PhysHlth", "MentHlth"] if c in df.columns]
    if health_limitation_cols:
        df["Health_Limitation_Score"] = df[health_limitation_cols].fillna(0).sum(axis=1)

    # 5. Socioeconomic proxy
    if {"Income", "Education"}.issubset(df.columns):
        df["SES"] = df["Income"] + df["Education"]

    return df

df_fe = add_engineered_features(df_clean)
print("Shape before feature engineering:", df_clean.shape)
print("Shape after feature engineering:", df_fe.shape)

new_columns = [c for c in df_fe.columns if c not in df_clean.columns]
print("Engineered features added:")
print(new_columns)

display(df_fe.head())


## 12. Feature Ranking and Feature Selection

The uploaded notebooks used multiple feature ranking/selection ideas:

- ANOVA F-score (`f_classif`)
- mutual information
- absolute target correlation
- SelectKBest for Logistic Regression
- Random Forest embedded feature importance

In [ ]:
def prepare_xy_multiclass(df, target_col="Diabetes_012"):
    # Prepare multiclass X/y without target leakage.
    X = df.drop(columns=[target_col, "Diabetes_binary"], errors="ignore")
    y = df[target_col].astype(int)
    return X, y

X_multi, y_multi = prepare_xy_multiclass(df_fe, target_col=target_col_original)

# Numeric conversion and median imputation used in tree-boosting notebooks
X_multi_numeric = X_multi.apply(pd.to_numeric, errors="coerce")
X_multi_numeric = X_multi_numeric.fillna(X_multi_numeric.median(numeric_only=True))

anova_f_values, anova_p_values = f_classif(X_multi_numeric, y_multi)
mi_values = mutual_info_classif(X_multi_numeric, y_multi, discrete_features="auto", random_state=RANDOM_STATE)

rank_frame = X_multi_numeric.copy()
rank_frame[target_col_original] = y_multi.values
target_corr = rank_frame.corr(numeric_only=True)[target_col_original].drop(target_col_original)

feature_ranking = pd.DataFrame({
    "feature": X_multi_numeric.columns,
    "anova_f": anova_f_values,
    "anova_p_value": anova_p_values,
    "mutual_information": mi_values,
    "target_correlation": target_corr.reindex(X_multi_numeric.columns).values,
})
feature_ranking["abs_target_correlation"] = feature_ranking["target_correlation"].abs()
feature_ranking = feature_ranking.sort_values("mutual_information", ascending=False).reset_index(drop=True)

display(feature_ranking)

fig, axes = plt.subplots(1, 3, figsize=(22, 6))
sns.barplot(data=feature_ranking.sort_values("anova_f", ascending=False).head(15), x="anova_f", y="feature", ax=axes[0])
axes[0].set_title("Top ANOVA F Features")

sns.barplot(data=feature_ranking.sort_values("mutual_information", ascending=False).head(15), x="mutual_information", y="feature", ax=axes[1])
axes[1].set_title("Top Mutual Information Features")

sns.barplot(data=feature_ranking.sort_values("abs_target_correlation", ascending=False).head(15), x="abs_target_correlation", y="feature", ax=axes[2])
axes[2].set_title("Top Absolute Target Correlations")

plt.tight_layout()
plt.show()


## 13. Train/Test Splits Used Across the Notebooks

The notebooks used reproducible stratified splits, commonly with `random_state=42`:

- 80/20 train/test for most classical ML notebooks
- 80/20 test split plus an internal validation split for neural models
- Random Forest notebook also used train/temp/test style in early cells

In [ ]:
# Multiclass split used by XGBoost / LightGBM / ExtraTrees / CatBoost / KNN-style notebooks
X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X_multi_numeric,
    y_multi,
    test_size=0.20,
    stratify=y_multi,
    random_state=RANDOM_STATE,
)

print("Multiclass split:")
print("X_train_multi:", X_train_multi.shape)
print("X_test_multi :", X_test_multi.shape)
print("Training distribution:")
display(y_train_multi.value_counts().sort_index().rename("count"))
print("Testing distribution:")
display(y_test_multi.value_counts().sort_index().rename("count"))

# Binary split used in Diabetes_XAI_hany_FINAL / Random Forest binary versions
df_binary_fe = df_fe.copy()
df_binary_fe["Diabetes_binary"] = (df_binary_fe[target_col_original] > 0).astype(int)
X_binary = df_binary_fe.drop(columns=[target_col_original, "Diabetes_binary"], errors="ignore").apply(pd.to_numeric, errors="coerce")
X_binary = X_binary.fillna(X_binary.median(numeric_only=True))
y_binary = df_binary_fe["Diabetes_binary"].astype(int)

X_trv_bin, X_test_bin, y_trv_bin, y_test_bin = train_test_split(
    X_binary,
    y_binary,
    test_size=0.20,
    stratify=y_binary,
    random_state=RANDOM_STATE,
)

X_train_bin, X_val_bin, y_train_bin, y_val_bin = train_test_split(
    X_trv_bin,
    y_trv_bin,
    test_size=0.10,
    stratify=y_trv_bin,
    random_state=RANDOM_STATE,
)

print("\nBinary train/validation/test split:")
print("X_train_bin:", X_train_bin.shape, "positive rate:", round(y_train_bin.mean() * 100, 2), "%")
print("X_val_bin  :", X_val_bin.shape, "positive rate:", round(y_val_bin.mean() * 100, 2), "%")
print("X_test_bin :", X_test_bin.shape, "positive rate:", round(y_test_bin.mean() * 100, 2), "%")
print("Imbalance ratio n_negative/n_positive:", round((y_train_bin == 0).sum() / (y_train_bin == 1).sum(), 3))


## 14. Scaling and Imputation Variants

The uploaded notebooks used different preprocessing variants depending on the model:

- **StandardScaler:** neural models, Logistic Regression, Random Forest pipeline
- **RobustScaler:** KNN/CatBoost section in Final_XAI
- **Median imputation without scaling:** XGBoost, LightGBM, ExtraTrees
- **SimpleImputer + ColumnTransformer:** Random Forest pipeline pattern

In [ ]:
# StandardScaler version
standard_scaler = StandardScaler()
X_train_multi_standard = pd.DataFrame(
    standard_scaler.fit_transform(X_train_multi),
    columns=X_train_multi.columns,
    index=X_train_multi.index,
)
X_test_multi_standard = pd.DataFrame(
    standard_scaler.transform(X_test_multi),
    columns=X_test_multi.columns,
    index=X_test_multi.index,
)

# RobustScaler version, used in Final_XAI for distance-based / CatBoost preprocessing section
robust_scaler = RobustScaler()
X_train_multi_robust = pd.DataFrame(
    robust_scaler.fit_transform(X_train_multi),
    columns=X_train_multi.columns,
    index=X_train_multi.index,
)
X_test_multi_robust = pd.DataFrame(
    robust_scaler.transform(X_test_multi),
    columns=X_test_multi.columns,
    index=X_test_multi.index,
)

# MinMaxScaler version kept as an optional normalization alternative
minmax_scaler = MinMaxScaler()
X_train_multi_minmax = pd.DataFrame(
    minmax_scaler.fit_transform(X_train_multi),
    columns=X_train_multi.columns,
    index=X_train_multi.index,
)
X_test_multi_minmax = pd.DataFrame(
    minmax_scaler.transform(X_test_multi),
    columns=X_test_multi.columns,
    index=X_test_multi.index,
)

print("Standard-scaled train shape:", X_train_multi_standard.shape)
print("Robust-scaled train shape  :", X_train_multi_robust.shape)
print("MinMax-scaled train shape  :", X_train_multi_minmax.shape)

# ColumnTransformer-style preprocessing pattern used in the Random Forest notebook
binary_features = [col for col in X_multi.columns if X_multi[col].dropna().nunique() <= 2]
numeric_features = [col for col in X_multi.columns if col not in binary_features]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

binary_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("bin", binary_transformer, binary_features),
    ],
    remainder="drop",
)

print("Numeric features:", len(numeric_features))
print("Binary/categorical-like features:", len(binary_features))


## 15. SMOTE and BorderlineSMOTE Balancing

SMOTE was applied only on the training split in the uploaded notebooks. This is the correct approach because oversampling the test set would leak synthetic information into evaluation.

In [ ]:
if IMBLEARN_AVAILABLE:
    # Multiclass SMOTE used by XGBoost/LightGBM/ExtraTrees notebooks
    smote_multi = SMOTE(sampling_strategy="not majority", k_neighbors=5, random_state=RANDOM_STATE)
    X_train_multi_smote, y_train_multi_smote = smote_multi.fit_resample(X_train_multi, y_train_multi)
    X_train_multi_smote = pd.DataFrame(X_train_multi_smote, columns=X_train_multi.columns)
    y_train_multi_smote = pd.Series(y_train_multi_smote, name=target_col_original)

    print("Multiclass training distribution before SMOTE:")
    display(y_train_multi.value_counts().sort_index().rename("count"))
    print("Multiclass training distribution after SMOTE:")
    display(y_train_multi_smote.value_counts().sort_index().rename("count"))

    # Standard SMOTE used in CatBoost experiments
    smote_standard = SMOTE(random_state=RANDOM_STATE)
    X_train_standard_smote, y_train_standard_smote = smote_standard.fit_resample(X_train_multi_robust, y_train_multi)
    X_train_standard_smote = pd.DataFrame(X_train_standard_smote, columns=X_train_multi_robust.columns)
    y_train_standard_smote = pd.Series(y_train_standard_smote, name=target_col_original)

    # BorderlineSMOTE experiment used in Final_XAI CatBoost section
    borderline_smote = BorderlineSMOTE(random_state=RANDOM_STATE)
    X_train_borderline_smote, y_train_borderline_smote = borderline_smote.fit_resample(X_train_multi_robust, y_train_multi)
    X_train_borderline_smote = pd.DataFrame(X_train_borderline_smote, columns=X_train_multi_robust.columns)
    y_train_borderline_smote = pd.Series(y_train_borderline_smote, name=target_col_original)

    print("\nBorderlineSMOTE distribution:")
    display(y_train_borderline_smote.value_counts().sort_index().rename("count"))
else:
    print("Skipping SMOTE cells because imbalanced-learn is unavailable.")


## 16. SelectKBest Feature Selection for Logistic Regression

The Logistic Regression notebook selected the top `k=15` features using ANOVA F-score after scaling.

In [ ]:
k = min(15, X_train_multi_standard.shape[1])

anova_selector = SelectKBest(score_func=f_classif, k=k)
X_train_selected = anova_selector.fit_transform(X_train_multi_standard, y_train_multi)
X_test_selected = anova_selector.transform(X_test_multi_standard)

selected_mask = anova_selector.get_support()
selected_features_logreg = X_train_multi_standard.columns[selected_mask].tolist()

X_train_selected_df = pd.DataFrame(
    X_train_selected,
    columns=selected_features_logreg,
    index=X_train_multi_standard.index,
)
X_test_selected_df = pd.DataFrame(
    X_test_selected,
    columns=selected_features_logreg,
    index=X_test_multi_standard.index,
)

print("Shape after SelectKBest:", X_train_selected_df.shape)
print("Selected Logistic Regression features:")
print(selected_features_logreg)


## 17. Random Forest Embedded Feature Selection

The Random Forest notebook engineered additional features and then trained a Random Forest feature selector. The top 10 features were retained.

In [ ]:
rf_feature_selector = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

rf_feature_selector.fit(X_train_multi, y_train_multi)

rf_feature_scores = pd.DataFrame({
    "Feature": X_train_multi.columns,
    "Random Forest Importance": rf_feature_selector.feature_importances_,
}).sort_values("Random Forest Importance", ascending=False).reset_index(drop=True)

k_rf = min(10, X_train_multi.shape[1])
selected_features_rf = rf_feature_scores.head(k_rf)["Feature"].tolist()

X_train_rf_selected = X_train_multi[selected_features_rf].copy()
X_test_rf_selected = X_test_multi[selected_features_rf].copy()

print("Random Forest selected features:")
print(selected_features_rf)

display(rf_feature_scores)

plt.figure(figsize=(10, 6))
sns.barplot(data=rf_feature_scores.head(15), x="Random Forest Importance", y="Feature")
plt.title("Random Forest Embedded Feature Importance")
plt.tight_layout()
plt.show()


## 18. Model-Specific Prepared Datasets

This table summarizes the final preprocessing choices available from this consolidated notebook.

In [ ]:
preprocessing_summary = pd.DataFrame([
    {
        "Prepared object": "X_train_multi / X_test_multi",
        "Target": "Diabetes_012 multiclass",
        "Cleaning / preprocessing": "numeric conversion + median imputation",
        "Used for": "XGBoost, LightGBM, ExtraTrees style tree models",
    },
    {
        "Prepared object": "X_train_multi_smote / y_train_multi_smote",
        "Target": "Diabetes_012 multiclass",
        "Cleaning / preprocessing": "SMOTE applied to training only",
        "Used for": "XGBoost, LightGBM, ExtraTrees, CatBoost SMOTE experiments",
    },
    {
        "Prepared object": "X_train_multi_standard / X_test_multi_standard",
        "Target": "Diabetes_012 multiclass",
        "Cleaning / preprocessing": "StandardScaler",
        "Used for": "Logistic Regression and models requiring standardized features",
    },
    {
        "Prepared object": "X_train_multi_robust / X_test_multi_robust",
        "Target": "Diabetes_012 multiclass",
        "Cleaning / preprocessing": "RobustScaler",
        "Used for": "KNN / CatBoost preprocessing branch from Final_XAI",
    },
    {
        "Prepared object": "X_train_selected_df / X_test_selected_df",
        "Target": "Diabetes_012 multiclass",
        "Cleaning / preprocessing": "StandardScaler + SelectKBest ANOVA top 15",
        "Used for": "Logistic Regression notebook",
    },
    {
        "Prepared object": "X_train_rf_selected / X_test_rf_selected",
        "Target": "Diabetes_012 multiclass",
        "Cleaning / preprocessing": "Random Forest embedded top-10 feature selection",
        "Used for": "Random Forest notebook",
    },
    {
        "Prepared object": "X_train_bin / X_val_bin / X_test_bin",
        "Target": "Diabetes_binary",
        "Cleaning / preprocessing": "binary target + train/validation/test split",
        "Used for": "MLP, TabNet, 1D-CNN, binary XAI notebook",
    },
])

display(preprocessing_summary)


## 19. Optional: Export Prepared Files

Run this cell if you want CSV versions of the processed datasets for the modeling notebooks.

In [ ]:
EXPORT_PROCESSED_FILES = False

if EXPORT_PROCESSED_FILES:
    output_dir = "processed_diabetes_data"
    os.makedirs(output_dir, exist_ok=True)

    df_clean.to_csv(os.path.join(output_dir, "df_clean.csv"), index=False)
    df_fe.to_csv(os.path.join(output_dir, "df_feature_engineered.csv"), index=False)

    X_train_multi.to_csv(os.path.join(output_dir, "X_train_multi.csv"), index=False)
    X_test_multi.to_csv(os.path.join(output_dir, "X_test_multi.csv"), index=False)
    y_train_multi.to_csv(os.path.join(output_dir, "y_train_multi.csv"), index=False)
    y_test_multi.to_csv(os.path.join(output_dir, "y_test_multi.csv"), index=False)

    X_train_multi_standard.to_csv(os.path.join(output_dir, "X_train_multi_standard.csv"), index=False)
    X_test_multi_standard.to_csv(os.path.join(output_dir, "X_test_multi_standard.csv"), index=False)

    X_train_selected_df.to_csv(os.path.join(output_dir, "X_train_selected_logreg.csv"), index=False)
    X_test_selected_df.to_csv(os.path.join(output_dir, "X_test_selected_logreg.csv"), index=False)

    X_train_rf_selected.to_csv(os.path.join(output_dir, "X_train_selected_rf.csv"), index=False)
    X_test_rf_selected.to_csv(os.path.join(output_dir, "X_test_selected_rf.csv"), index=False)

    print("Exported processed files to:", output_dir)
else:
    print("Set EXPORT_PROCESSED_FILES = True to export processed datasets.")


## 20. Source Mapping

This notebook covers the following EDA, preprocessing, and feature-engineering actions from the uploaded notebooks:

| Source notebook | Consolidated actions |
|---|---|
| `Final_XAI` | `head`, `info`, `describe`, missing values, class distribution, BMI IQR outliers, group means, correlation, skewness, countplots, heatmap, boxplots, KDEs, duplicate removal, BMI outlier removal, `BMI_Age`, correlation-based feature selection, stratified train/test split, RobustScaler, SMOTE, BorderlineSMOTE |
| `Diabetes_XAI_hany_FINAL` | binary target creation, `dropna`, class distribution, binary correlation heatmap, continuous KDE by binary class, `BMI_x_Age`, `Hlth_burden`, `Risk_score`, `Lifestyle_score`, `SES`, ANOVA F-score ranking, train/validation/test split, StandardScaler |
| `Ibrahim Atef_DECISION TREE` | `BMI_Age`, `Health_Risk_Score`, `Lifestyle_Score`, `Poor_Health_Days` |
| `Ibrahim Atef_LOGISTIC Regression` | scaling, ANOVA `SelectKBest`, selected feature dataframes |
| `Ibrahim Atef_RANDOM_FORESt` | missing/drop cleaning, binary target, leakage-safe feature separation, class distribution, descriptive statistics, target correlations, distribution plots, binary/numeric feature detection, preprocessing pipeline, `Cardiometabolic_Risk_Score`, inverse lifestyle indicators, `Lifestyle_Risk_Score`, `Health_Limitation_Score`, `BMI_Age_Interaction`, RF embedded feature selection |
| `Mohammed_Sherbini_XGBoost` | lowercase target handling, stratified split, numeric conversion, median imputation, SMOTE `not majority`, ANOVA, mutual information, correlation ranking |
| `Mohammed_Sherbini_LightGBM` | same preprocessing/ranking pattern as XGBoost with LightGBM modeling branch |
| `Mohammed_Sherbini_ExtraTrees` | same preprocessing/ranking pattern as XGBoost with ExtraTrees modeling branch |